In [1]:
# ============================================================
# MATHDIAL KAGGLE INPUT VALIDATION
# ============================================================

import json
from pathlib import Path
from collections import Counter


LABELS = {
    "generic",
    "probing",
    "focus",
    "telling",
}


print("=" * 70)
print("MATHDIAL KAGGLE INPUT VALIDATION")
print("=" * 70)


# ============================================================
# FIND UPLOADED FILES
# ============================================================

input_root = Path("/kaggle/input")

train_candidates = list(
    input_root.rglob("train.jsonl")
)

val_candidates = list(
    input_root.rglob("validation.jsonl")
)


print("\ntrain.jsonl candidates:")
for path in train_candidates:
    print(" ", path)


print("\nvalidation.jsonl candidates:")
for path in val_candidates:
    print(" ", path)


if len(train_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one train.jsonl, "
        f"found {len(train_candidates)}"
    )


if len(val_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one validation.jsonl, "
        f"found {len(val_candidates)}"
    )


TRAIN_PATH = train_candidates[0]
VAL_PATH = val_candidates[0]


print(
    f"\nTRAIN_PATH:\n{TRAIN_PATH}"
)

print(
    f"\nVAL_PATH:\n{VAL_PATH}"
)


# ============================================================
# LOAD JSONL
# ============================================================

def load_jsonl(path):

    rows = []

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


train_rows = load_jsonl(
    TRAIN_PATH
)

val_rows = load_jsonl(
    VAL_PATH
)


print(
    f"\nTrain examples: "
    f"{len(train_rows)}"
)

print(
    f"Validation examples: "
    f"{len(val_rows)}"
)


# ============================================================
# EXPECTED SIZE
# ============================================================

assert len(train_rows) == 14905, (
    f"Expected 14905 training examples, "
    f"found {len(train_rows)}"
)

assert len(val_rows) == 1850, (
    f"Expected 1850 validation examples, "
    f"found {len(val_rows)}"
)


# ============================================================
# STRUCTURE VALIDATION
# ============================================================

required_fields = {
    "example_id",
    "dialogue_id",
    "qid",
    "group_id",
    "problem",
    "history",
    "history_num_turns",
    "target_move",
}


def validate_rows(name, rows):

    missing_fields = 0
    invalid_labels = 0
    history_errors = 0
    dialog_act_leakage = 0


    for row in rows:

        if not required_fields.issubset(
            row.keys()
        ):
            missing_fields += 1


        if row.get(
            "target_move"
        ) not in LABELS:
            invalid_labels += 1


        if len(
            row.get("history", [])
        ) != row.get(
            "history_num_turns"
        ):
            history_errors += 1


        for turn in row.get(
            "history",
            []
        ):

            if "dialog_act" in turn:
                dialog_act_leakage += 1


    print(
        f"\n{name}:"
    )

    print(
        "  Missing fields:",
        missing_fields
    )

    print(
        "  Invalid labels:",
        invalid_labels
    )

    print(
        "  History errors:",
        history_errors
    )

    print(
        "  Historical dialog_act leakage:",
        dialog_act_leakage
    )


    assert missing_fields == 0
    assert invalid_labels == 0
    assert history_errors == 0
    assert dialog_act_leakage == 0


validate_rows(
    "TRAIN",
    train_rows
)

validate_rows(
    "VALIDATION",
    val_rows
)


# ============================================================
# LABEL DISTRIBUTION
# ============================================================

def print_distribution(name, rows):

    counts = Counter(
        row["target_move"]
        for row in rows
    )


    print(
        f"\n{name} label distribution:"
    )


    for label in [
        "generic",
        "probing",
        "focus",
        "telling",
    ]:

        count = counts[label]

        percentage = (
            count
            / len(rows)
            * 100
        )

        print(
            f"  {label:10s}: "
            f"{count:5d} "
            f"({percentage:.2f}%)"
        )


print_distribution(
    "TRAIN",
    train_rows
)

print_distribution(
    "VALIDATION",
    val_rows
)


# ============================================================
# LEAKAGE CHECK
# ============================================================

def normalize_problem(text):

    return " ".join(
        text.lower().split()
    )


train_qids = {
    row["qid"]
    for row in train_rows
}

val_qids = {
    row["qid"]
    for row in val_rows
}


train_groups = {
    row["group_id"]
    for row in train_rows
}

val_groups = {
    row["group_id"]
    for row in val_rows
}


train_problems = {
    normalize_problem(
        row["problem"]
    )
    for row in train_rows
}

val_problems = {
    normalize_problem(
        row["problem"]
    )
    for row in val_rows
}


qid_overlap = len(
    train_qids
    & val_qids
)

group_overlap = len(
    train_groups
    & val_groups
)

problem_overlap = len(
    train_problems
    & val_problems
)


print(
    "\n"
    + "=" * 70
)

print(
    "TRAIN / VALIDATION LEAKAGE"
)

print(
    "=" * 70
)


print(
    "\nQID overlap:",
    qid_overlap
)

print(
    "Group overlap:",
    group_overlap
)

print(
    "Problem overlap:",
    problem_overlap
)


assert qid_overlap == 0
assert group_overlap == 0
assert problem_overlap == 0


print(
    "\n"
    + "=" * 70
)

print(
    "KAGGLE INPUT VALIDATION: PASSED"
)

print(
    "=" * 70
)

MATHDIAL KAGGLE INPUT VALIDATION

train.jsonl candidates:
  /kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl

validation.jsonl candidates:
  /kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl

TRAIN_PATH:
/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl

VAL_PATH:
/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl

Train examples: 14905
Validation examples: 1850

TRAIN:
  Missing fields: 0
  Invalid labels: 0
  History errors: 0
  Historical dialog_act leakage: 0

VALIDATION:
  Missing fields: 0
  Invalid labels: 0
  History errors: 0
  Historical dialog_act leakage: 0

TRAIN label distribution:
  generic   :  3579 (24.01%)
  probing   :  3413 (22.90%)
  focus     :  5443 (36.52%)
  telling   :  2470 (16.57%)

VALIDATION label distribution:
  generic   :   442 (23.89%)
  probing   :   412 (22.27%)
  focus     :   678 (36.65%)
  telling   :   318 (17.19%)

TRAIN / VALIDATION LEAKAGE

QID overlap: 0
Group o

In [2]:
# ============================================================
# MATHDIAL TOKEN-LENGTH + TRUNCATION VALIDATION
#
# Goal:
#   Preserve:
#     1. the mathematics problem
#     2. the MOST RECENT conversation turns
#
# while staying within DistilRoBERTa's 512-token limit.
# ============================================================

import numpy as np

from transformers import AutoTokenizer


MODEL_NAME = "distilbert/distilroberta-base"

MAX_LENGTH = 512


print("=" * 70)
print("MATHDIAL TOKENIZATION / TRUNCATION VALIDATION")
print("=" * 70)


# ============================================================
# LOAD TOKENIZER
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# IMPORTANT:
# When the conversation is too long,
# truncate OLD conversation tokens first.
tokenizer.truncation_side = "left"


print(
    "\nTokenizer:",
    tokenizer.__class__.__name__
)

print(
    "Truncation side:",
    tokenizer.truncation_side
)

print(
    "Model max length:",
    tokenizer.model_max_length
)


# ============================================================
# INPUT FORMAT
# ============================================================

def format_history(history):

    if not history:

        return "[No previous conversation]"

    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )


def build_problem_text(example):

    return (
        "Problem:\n"
        f"{example['problem']}"
    )


def build_conversation_text(example):

    return (
        "Conversation:\n"
        f"{format_history(example['history'])}"
        "\n\n"
        "Next teacher pedagogical move:"
    )


# ============================================================
# MEASURE ORIGINAL TOKEN LENGTHS
# ============================================================

all_rows = (
    train_rows
    + val_rows
)


full_lengths = []
problem_lengths = []

over_512 = 0


for example in all_rows:

    problem_text = (
        build_problem_text(example)
    )

    conversation_text = (
        build_conversation_text(example)
    )


    problem_ids = tokenizer(
        problem_text,
        add_special_tokens=False,
    )["input_ids"]


    full_ids = tokenizer(
        problem_text,
        conversation_text,
        add_special_tokens=True,
        truncation=False,
    )["input_ids"]


    problem_lengths.append(
        len(problem_ids)
    )

    full_lengths.append(
        len(full_ids)
    )


    if len(full_ids) > MAX_LENGTH:

        over_512 += 1


full_lengths_np = np.array(
    full_lengths
)

problem_lengths_np = np.array(
    problem_lengths
)


print(
    "\n"
    + "=" * 70
)

print(
    "UNTRUNCATED TOKEN LENGTHS"
)

print(
    "=" * 70
)


print(
    f"\nExamples checked: "
    f"{len(all_rows)}"
)

print(
    f"Mean full length: "
    f"{full_lengths_np.mean():.2f}"
)

print(
    f"Median full length: "
    f"{np.median(full_lengths_np):.2f}"
)

print(
    f"95th percentile: "
    f"{np.percentile(full_lengths_np, 95):.2f}"
)

print(
    f"99th percentile: "
    f"{np.percentile(full_lengths_np, 99):.2f}"
)

print(
    f"Maximum full length: "
    f"{full_lengths_np.max()}"
)


print(
    f"\nExamples exceeding 512 tokens: "
    f"{over_512}"
)

print(
    f"Percentage exceeding 512: "
    f"{(
        over_512
        / len(all_rows)
        * 100
    ):.2f}%"
)


print(
    f"\nMaximum problem-only length: "
    f"{problem_lengths_np.max()}"
)


# ============================================================
# VALIDATE PROBLEM CAN ALWAYS FIT
# ============================================================

too_long_problem_count = sum(

    length >= 500

    for length in problem_lengths
)


print(
    f"Problems >= 500 tokens: "
    f"{too_long_problem_count}"
)


# ============================================================
# APPLY DESIRED TRUNCATION
#
# truncation="only_second":
#
#   first sequence  = problem
#   second sequence = conversation
#
# Together with:
#
#   truncation_side="left"
#
# this removes OLD conversation tokens while preserving
# the problem and newest conversation context.
# ============================================================

truncated_lengths = []

truncation_failures = 0


for example in all_rows:

    encoded = tokenizer(

        build_problem_text(example),

        build_conversation_text(example),

        truncation="only_second",

        max_length=MAX_LENGTH,

        add_special_tokens=True,
    )


    length = len(
        encoded["input_ids"]
    )


    truncated_lengths.append(
        length
    )


    if length > MAX_LENGTH:

        truncation_failures += 1


print(
    "\n"
    + "=" * 70
)

print(
    "POST-TRUNCATION VALIDATION"
)

print(
    "=" * 70
)


print(
    f"\nMaximum encoded length: "
    f"{max(truncated_lengths)}"
)

print(
    f"Sequences >512 after truncation: "
    f"{truncation_failures}"
)


# ============================================================
# VERIFY RECENT TURN PRESERVATION
#
# For examples requiring truncation, check whether the
# last observable conversation turn is still represented.
# ============================================================

truncated_examples = [

    (
        example,
        original_length
    )

    for example, original_length
    in zip(
        all_rows,
        full_lengths
    )

    if original_length > MAX_LENGTH
]


recent_turn_checks = 0
recent_turn_preserved = 0


for example, _ in truncated_examples[:100]:

    if not example["history"]:
        continue


    last_turn = (
        example["history"][-1]
    )


    last_turn_text = (
        f"{last_turn['user']}: "
        f"{last_turn['text']}"
    )


    # Use only the final 12 tokens of the latest turn.
    # This makes the check robust when a single turn itself
    # is unusually long.

    last_turn_ids = tokenizer(
        last_turn_text,
        add_special_tokens=False,
    )["input_ids"]


    suffix_ids = (
        last_turn_ids[-12:]
    )


    encoded = tokenizer(

        build_problem_text(example),

        build_conversation_text(example),

        truncation="only_second",

        max_length=MAX_LENGTH,

        add_special_tokens=True,
    )["input_ids"]


    recent_turn_checks += 1


    # Search for suffix token sequence
    # inside final encoded input.

    found = False


    for start in range(
        0,
        len(encoded)
        - len(suffix_ids)
        + 1
    ):

        if (
            encoded[
                start:
                start + len(suffix_ids)
            ]
            == suffix_ids
        ):

            found = True
            break


    if found:

        recent_turn_preserved += 1


print(
    f"\nRecent-turn checks performed: "
    f"{recent_turn_checks}"
)

print(
    f"Recent turns preserved: "
    f"{recent_turn_preserved}"
)


if recent_turn_checks:

    preservation_rate = (
        recent_turn_preserved
        / recent_turn_checks
        * 100
    )

else:

    preservation_rate = 100.0


print(
    f"Recent-turn preservation rate: "
    f"{preservation_rate:.2f}%"
)


# ============================================================
# FINAL
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "TOKENIZATION VALIDATION SUMMARY"
)

print(
    "=" * 70
)


print(
    f"\nExamples >512 before truncation: "
    f"{over_512}"
)

print(
    f"Maximum problem length: "
    f"{problem_lengths_np.max()}"
)

print(
    f"Sequences >512 after truncation: "
    f"{truncation_failures}"
)

print(
    f"Recent-turn preservation rate: "
    f"{preservation_rate:.2f}%"
)


if (
    too_long_problem_count == 0
    and truncation_failures == 0
    and preservation_rate == 100.0
):

    print(
        "\nTOKENIZATION VALIDATION: PASSED"
    )

else:

    print(
        "\nTOKENIZATION VALIDATION: REVIEW REQUIRED"
    )

MATHDIAL TOKENIZATION / TRUNCATION VALIDATION


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (597 > 512). Running this sequence through the model will result in indexing errors



Tokenizer: RobertaTokenizer
Truncation side: left
Model max length: 512

UNTRUNCATED TOKEN LENGTHS

Examples checked: 16755
Mean full length: 335.52
Median full length: 288.00
95th percentile: 760.30
99th percentile: 1095.00
Maximum full length: 2134

Examples exceeding 512 tokens: 3061
Percentage exceeding 512: 18.27%

Maximum problem-only length: 175
Problems >= 500 tokens: 0

POST-TRUNCATION VALIDATION

Maximum encoded length: 512
Sequences >512 after truncation: 0

Recent-turn checks performed: 100
Recent turns preserved: 100
Recent-turn preservation rate: 100.00%

TOKENIZATION VALIDATION SUMMARY

Examples >512 before truncation: 3061
Maximum problem length: 175
Sequences >512 after truncation: 0
Recent-turn preservation rate: 100.00%

TOKENIZATION VALIDATION: PASSED


In [3]:
# ============================================================
# MD2 — MATHDIAL DISTILROBERTA MOVE SELECTOR
#
# TRAIN:
#   MathDial train = 14,905 examples
#
# VALIDATE:
#   MathDial validation = 1,850 examples
#
# INPUT:
#   Problem + observable conversation history
#
# OUTPUT:
#   generic / probing / focus / telling
#
# LOSS:
#   Ordinary Cross-Entropy
#
# LONG CONTEXT:
#   Preserve problem + newest conversation tokens.
#
# TEST SET:
#   NOT USED.
# ============================================================

import gc
import json
import random
from pathlib import Path

import numpy as np
import torch

from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


# ============================================================
# CONFIG
# ============================================================

SEED = 42

MODEL_NAME = (
    "distilbert/distilroberta-base"
)

TRAIN_PATH = Path(
    "/kaggle/input/datasets/"
    "malindus/mathdial-move-selection/"
    "train.jsonl"
)

VAL_PATH = Path(
    "/kaggle/input/datasets/"
    "malindus/mathdial-move-selection/"
    "validation.jsonl"
)

OUTPUT_DIR = (
    "/kaggle/working/"
    "md2_distilroberta"
)

RESULT_PATH = Path(
    "/kaggle/working/"
    "md2_distilroberta_validation.json"
)

PREDICTION_PATH = Path(
    "/kaggle/working/"
    "md2_validation_predictions.jsonl"
)


MAX_LENGTH = 512


LABELS = [
    "generic",
    "probing",
    "focus",
    "telling",
]


LABEL2ID = {
    label: index
    for index, label
    in enumerate(LABELS)
}


ID2LABEL = {
    index: label
    for label, index
    in LABEL2ID.items()
}


# ============================================================
# REPRODUCIBILITY
# ============================================================

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# START
# ============================================================

print("=" * 70)

print(
    "MD2 — DISTILROBERTA + ORDINARY CROSS-ENTROPY"
)

print("=" * 70)


print(
    "\nCUDA available:",
    torch.cuda.is_available()
)

print(
    "GPU count:",
    torch.cuda.device_count()
)


for index in range(
    torch.cuda.device_count()
):

    print(
        f"GPU {index}:",
        torch.cuda.get_device_name(
            index
        )
    )


# ============================================================
# CLEAR PREVIOUS MODEL OBJECTS
# ============================================================

for object_name in [
    "model",
    "trainer",
    "clean_trainer",
    "fresh_model",
    "state_model",
]:

    if object_name in globals():

        del globals()[object_name]


gc.collect()


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# LOAD DATA
# ============================================================

def load_jsonl(path):

    rows = []

    with path.open(
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:

                rows.append(
                    json.loads(line)
                )

    return rows


train_rows_md2 = load_jsonl(
    TRAIN_PATH
)

val_rows_md2 = load_jsonl(
    VAL_PATH
)


print(
    f"\nTrain examples: "
    f"{len(train_rows_md2)}"
)

print(
    f"Validation examples: "
    f"{len(val_rows_md2)}"
)


assert (
    len(train_rows_md2)
    == 14905
)

assert (
    len(val_rows_md2)
    == 1850
)


# ============================================================
# FORMAT INPUT
# ============================================================

def format_history(history):

    if not history:

        return (
            "[No previous conversation]"
        )


    return "\n".join(

        f"{turn['user']}: "
        f"{turn['text']}"

        for turn in history
    )


def build_problem_text(
    example
):

    return (
        "Problem:\n"
        f"{example['problem']}"
    )


def build_conversation_text(
    example
):

    return (
        "Conversation:\n"
        f"{format_history(example['history'])}"
        "\n\n"
        "Next teacher pedagogical move:"
    )


# ============================================================
# TOKENIZER
# ============================================================

print(
    f"\nLoading tokenizer:"
    f"\n{MODEL_NAME}"
)


tokenizer_md2 = (
    AutoTokenizer
    .from_pretrained(
        MODEL_NAME
    )
)


# Critical:
# when truncating conversation,
# remove OLD context first.

tokenizer_md2.truncation_side = (
    "left"
)


print(
    "Truncation side:",
    tokenizer_md2.truncation_side
)


# ============================================================
# PREPARE DATASET
# ============================================================

def prepare_examples(
    rows
):

    prepared = []


    for example in rows:

        prepared.append({

            "example_id":
                example[
                    "example_id"
                ],

            "problem_text":
                build_problem_text(
                    example
                ),

            "conversation_text":
                build_conversation_text(
                    example
                ),

            "labels":
                LABEL2ID[
                    example[
                        "target_move"
                    ]
                ],
        })


    return prepared


prepared_train = (
    prepare_examples(
        train_rows_md2
    )
)

prepared_val = (
    prepare_examples(
        val_rows_md2
    )
)


train_dataset_raw = (
    Dataset.from_list(
        prepared_train
    )
)


val_dataset_raw = (
    Dataset.from_list(
        prepared_val
    )
)


# ============================================================
# CALCULATE ORIGINAL VALIDATION LENGTHS
#
# Used later to compare:
#   short      <= 512
#   truncated  > 512
# ============================================================

print(
    "\nCalculating original "
    "validation token lengths..."
)


val_original_lengths = []


for example in val_rows_md2:

    encoded = tokenizer_md2(

        build_problem_text(
            example
        ),

        build_conversation_text(
            example
        ),

        add_special_tokens=True,

        truncation=False,
    )


    val_original_lengths.append(
        len(
            encoded[
                "input_ids"
            ]
        )
    )


val_original_lengths = (
    np.array(
        val_original_lengths
    )
)


short_mask = (
    val_original_lengths
    <= MAX_LENGTH
)


long_mask = (
    val_original_lengths
    > MAX_LENGTH
)


print(
    f"\nValidation <=512: "
    f"{short_mask.sum()}"
)

print(
    f"Validation >512: "
    f"{long_mask.sum()}"
)

print(
    f"Validation truncated percentage: "
    f"{(
        long_mask.mean()
        * 100
    ):.2f}%"
)


# ============================================================
# TOKENIZATION
#
# First sequence:
#   problem
#
# Second sequence:
#   conversation
#
# only_second + truncation_side=left:
#   preserve problem,
#   drop OLD conversation context.
# ============================================================

def tokenize_batch(
    batch
):

    return tokenizer_md2(

        batch[
            "problem_text"
        ],

        batch[
            "conversation_text"
        ],

        truncation="only_second",

        max_length=MAX_LENGTH,
    )


print(
    "\nTokenizing training set..."
)


tokenized_train = (
    train_dataset_raw.map(

        tokenize_batch,

        batched=True,

        remove_columns=[
            "example_id",
            "problem_text",
            "conversation_text",
        ],
    )
)


print(
    "\nTokenizing validation set..."
)


tokenized_val = (
    val_dataset_raw.map(

        tokenize_batch,

        batched=True,

        remove_columns=[
            "example_id",
            "problem_text",
            "conversation_text",
        ],
    )
)


# ============================================================
# VERIFY ENCODED LENGTH
# ============================================================

train_encoded_lengths = [

    len(ids)

    for ids
    in tokenized_train[
        "input_ids"
    ]
]


val_encoded_lengths = [

    len(ids)

    for ids
    in tokenized_val[
        "input_ids"
    ]
]


print(
    f"\nMaximum encoded TRAIN length: "
    f"{max(train_encoded_lengths)}"
)

print(
    f"Maximum encoded VALIDATION length: "
    f"{max(val_encoded_lengths)}"
)


assert (
    max(train_encoded_lengths)
    <= MAX_LENGTH
)

assert (
    max(val_encoded_lengths)
    <= MAX_LENGTH
)


# ============================================================
# DYNAMIC PADDING
# ============================================================

data_collator = (
    DataCollatorWithPadding(
        tokenizer=tokenizer_md2
    )
)


# ============================================================
# MODEL
# ============================================================

print(
    "\nLoading DistilRoBERTa "
    "classification model..."
)


model_md2 = (
    AutoModelForSequenceClassification
    .from_pretrained(

        MODEL_NAME,

        num_labels=
            len(LABELS),

        label2id=
            LABEL2ID,

        id2label=
            ID2LABEL,
    )
)


# ============================================================
# METRICS
# ============================================================

def compute_metrics(
    eval_prediction
):

    logits, labels = (
        eval_prediction
    )


    predictions = np.argmax(
        logits,
        axis=-1
    )


    return {

        "accuracy":
            accuracy_score(
                labels,
                predictions
            ),

        "macro_f1":
            f1_score(
                labels,
                predictions,
                labels=list(
                    range(
                        len(LABELS)
                    )
                ),
                average="macro",
                zero_division=0,
            ),
    }


# ============================================================
# TRAINING CONFIG
#
# Plain CE:
# no weights
# no oversampling
# no focal loss
#
# 2 GPUs × batch 8 = global batch 16.
#
# ~932 optimizer steps / epoch
# ~2796 total for 3 epochs
# 280 warmup steps ≈ 10%
# ============================================================

training_args_md2 = (
    TrainingArguments(

        output_dir=
            OUTPUT_DIR,

        learning_rate=
            2e-5,

        per_device_train_batch_size=
            8,

        per_device_eval_batch_size=
            16,

        num_train_epochs=
            3,

        weight_decay=
            0.01,

        warmup_steps=
            280,

        eval_strategy=
            "epoch",

        logging_strategy=
            "epoch",

        save_strategy=
            "no",

        load_best_model_at_end=
            False,

        fp16=
            True,

        seed=
            SEED,

        data_seed=
            SEED,

        report_to=
            "none",
    )
)


# ============================================================
# TRAINER
# ============================================================

trainer_md2 = Trainer(

    model=
        model_md2,

    args=
        training_args_md2,

    train_dataset=
        tokenized_train,

    eval_dataset=
        tokenized_val,

    processing_class=
        tokenizer_md2,

    data_collator=
        data_collator,

    compute_metrics=
        compute_metrics,
)


print(
    "\nTrainer GPU count:",
    trainer_md2.args.n_gpu
)


# ============================================================
# TRAIN
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "STARTING MD2 TRAINING"
)

print(
    "=" * 70
)


trainer_md2.train()


print(
    "\nMD2 training complete."
)


# ============================================================
# SHOW EPOCH VALIDATION HISTORY
# ============================================================

epoch_evaluations = [

    log

    for log
    in trainer_md2.state.log_history

    if "eval_macro_f1" in log
]


print(
    "\n"
    + "=" * 70
)

print(
    "MD2 EPOCH VALIDATION HISTORY"
)

print(
    "=" * 70
)


for log in epoch_evaluations:

    print(
        f"\nEpoch: "
        f"{log.get('epoch')}"
    )

    print(
        f"  Validation loss: "
        f"{log.get('eval_loss'):.6f}"
    )

    print(
        f"  Accuracy: "
        f"{log.get('eval_accuracy'):.6f}"
    )

    print(
        f"  Macro-F1: "
        f"{log.get('eval_macro_f1'):.6f}"
    )


# ============================================================
# FINAL IN-MEMORY PREDICTION
# ============================================================

print(
    "\nRunning final validation prediction..."
)


prediction_output = (
    trainer_md2.predict(
        tokenized_val
    )
)


logits = (
    prediction_output.predictions
)


true_ids = (
    prediction_output.label_ids
)


pred_ids = np.argmax(
    logits,
    axis=-1
)


# ============================================================
# PROBABILITIES
# ============================================================

stable_logits = (
    logits
    - logits.max(
        axis=1,
        keepdims=True
    )
)


exp_logits = np.exp(
    stable_logits
)


probabilities = (
    exp_logits
    / exp_logits.sum(
        axis=1,
        keepdims=True
    )
)


# ============================================================
# EVALUATION FUNCTION
# ============================================================

def evaluate_subset(
    name,
    mask
):

    subset_true = (
        true_ids[mask]
    )

    subset_pred = (
        pred_ids[mask]
    )


    accuracy = accuracy_score(
        subset_true,
        subset_pred
    )


    macro_f1 = f1_score(

        subset_true,

        subset_pred,

        labels=list(
            range(
                len(LABELS)
            )
        ),

        average="macro",

        zero_division=0,
    )


    per_class = f1_score(

        subset_true,

        subset_pred,

        labels=list(
            range(
                len(LABELS)
            )
        ),

        average=None,

        zero_division=0,
    )


    matrix = confusion_matrix(

        subset_true,

        subset_pred,

        labels=list(
            range(
                len(LABELS)
            )
        ),
    )


    support = Counter(
        subset_true.tolist()
    )


    print(
        "\n"
        + "=" * 70
    )

    print(name)

    print("=" * 70)


    print(
        f"\nExamples: "
        f"{len(subset_true)}"
    )

    print(
        f"Accuracy: "
        f"{accuracy:.6f}"
    )

    print(
        f"Macro-F1: "
        f"{macro_f1:.6f}"
    )


    print(
        "\nPer-class F1:"
    )


    for label, score in zip(
        LABELS,
        per_class
    ):

        print(
            f"{label:10s}: "
            f"{score:.6f}"
        )


    print(
        "\nClass support:"
    )


    for class_id, label in enumerate(
        LABELS
    ):

        print(
            f"{label:10s}: "
            f"{support[class_id]}"
        )


    print(
        "\nConfusion Matrix:"
    )

    print(
        "Label order:",
        LABELS
    )

    print(matrix)


    return {

        "examples":
            int(
                len(subset_true)
            ),

        "accuracy":
            float(
                accuracy
            ),

        "macro_f1":
            float(
                macro_f1
            ),

        "per_class_f1": {

            label:
                float(score)

            for label, score
            in zip(
                LABELS,
                per_class
            )
        },

        "class_support": {

            label:
                int(
                    support[class_id]
                )

            for class_id, label
            in enumerate(
                LABELS
            )
        },

        "confusion_matrix":
            matrix.tolist(),
    }


# ============================================================
# OVERALL
# ============================================================

all_mask = np.ones(
    len(true_ids),
    dtype=bool
)


overall_result = (
    evaluate_subset(
        "MD2 — OVERALL VALIDATION",
        all_mask
    )
)


# ============================================================
# SHORT / NON-TRUNCATED
# ============================================================

short_result = (
    evaluate_subset(

        "MD2 — NON-TRUNCATED VALIDATION "
        "(ORIGINAL <=512 TOKENS)",

        short_mask,
    )
)


# ============================================================
# LONG / TRUNCATED
# ============================================================

long_result = (
    evaluate_subset(

        "MD2 — TRUNCATED VALIDATION "
        "(ORIGINAL >512 TOKENS)",

        long_mask,
    )
)


# ============================================================
# FULL CLASSIFICATION REPORT
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "MD2 OVERALL CLASSIFICATION REPORT"
)

print(
    "=" * 70
)


print(
    classification_report(

        true_ids,

        pred_ids,

        labels=list(
            range(
                len(LABELS)
            )
        ),

        target_names=
            LABELS,

        digits=4,

        zero_division=0,
    )
)


# ============================================================
# COMPARE WITH TF-IDF BASELINE
# ============================================================

TFIDF_MACRO_F1 = (
    0.405754
)


print(
    "\n"
    + "=" * 70
)

print(
    "MD2 vs MD1"
)

print(
    "=" * 70
)


print(
    f"\nMD1 TF-IDF Macro-F1: "
    f"{TFIDF_MACRO_F1:.6f}"
)

print(
    f"MD2 DistilRoBERTa Macro-F1: "
    f"{overall_result['macro_f1']:.6f}"
)


improvement = (
    overall_result[
        "macro_f1"
    ]
    - TFIDF_MACRO_F1
)


print(
    f"Improvement: "
    f"{improvement:+.6f}"
)


# ============================================================
# LONG-CONTEXT GAP
# ============================================================

context_gap = (

    short_result[
        "macro_f1"
    ]

    - long_result[
        "macro_f1"
    ]
)


print(
    "\n"
    + "=" * 70
)

print(
    "LONG-CONTEXT ANALYSIS"
)

print(
    "=" * 70
)


print(
    f"\n<=512 Macro-F1: "
    f"{short_result['macro_f1']:.6f}"
)

print(
    f">512 Macro-F1: "
    f"{long_result['macro_f1']:.6f}"
)

print(
    f"Short - Long gap: "
    f"{context_gap:+.6f}"
)


# ============================================================
# SAVE RESULTS
# ============================================================

results = {

    "experiment":
        "MD2_distilroberta_plain_ce",

    "model":
        MODEL_NAME,

    "epochs":
        3,

    "max_length":
        MAX_LENGTH,

    "truncation":
        "problem_preserved_recent_conversation",

    "loss":
        "ordinary_cross_entropy",

    "train_examples":
        len(
            train_rows_md2
        ),

    "validation_examples":
        len(
            val_rows_md2
        ),

    "validation_original_over_512":
        int(
            long_mask.sum()
        ),

    "overall":
        overall_result,

    "non_truncated":
        short_result,

    "truncated":
        long_result,

    "short_minus_long_macro_f1":
        float(
            context_gap
        ),

    "tfidf_macro_f1":
        TFIDF_MACRO_F1,

    "improvement_over_tfidf":
        float(
            improvement
        ),

    "epoch_history":
        epoch_evaluations,
}


with RESULT_PATH.open(
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2
    )


# ============================================================
# SAVE PER-EXAMPLE VALIDATION PREDICTIONS
#
# Useful later for local error analysis without rerunning GPU.
# ============================================================

with PREDICTION_PATH.open(
    "w",
    encoding="utf-8"
) as f:

    for index, example in enumerate(
        val_rows_md2
    ):

        prediction_record = {

            "example_id":
                example[
                    "example_id"
                ],

            "qid":
                example[
                    "qid"
                ],

            "history_num_turns":
                example[
                    "history_num_turns"
                ],

            "original_token_length":
                int(
                    val_original_lengths[
                        index
                    ]
                ),

            "was_truncated":
                bool(
                    long_mask[
                        index
                    ]
                ),

            "true_move":
                LABELS[
                    int(
                        true_ids[
                            index
                        ]
                    )
                ],

            "predicted_move":
                LABELS[
                    int(
                        pred_ids[
                            index
                        ]
                    )
                ],

            "confidence":
                float(
                    probabilities[
                        index
                    ].max()
                ),

            "probabilities": {

                label:
                    float(
                        probabilities[
                            index,
                            label_id
                        ]
                    )

                for label_id, label
                in enumerate(
                    LABELS
                )
            },
        }


        f.write(
            json.dumps(
                prediction_record,
                ensure_ascii=False
            )
            + "\n"
        )


print(
    "\n"
    + "=" * 70
)

print(
    "MD2 OUTPUT FILES"
)

print(
    "=" * 70
)


print(
    "\nResults:"
)

print(
    RESULT_PATH
)

print(
    "\nPer-example predictions:"
)

print(
    PREDICTION_PATH
)


print(
    "\n"
    + "=" * 70
)

print(
    "MD2 EXPERIMENT COMPLETE"
)

print(
    "=" * 70
)

MD2 — DISTILROBERTA + ORDINARY CROSS-ENTROPY

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Train examples: 14905
Validation examples: 1850

Loading tokenizer:
distilbert/distilroberta-base
Truncation side: left


Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors



Calculating original validation token lengths...

Validation <=512: 1507
Validation >512: 343
Validation truncated percentage: 18.54%

Tokenizing training set...


Map:   0%|          | 0/14905 [00:00<?, ? examples/s]


Tokenizing validation set...


...

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.402246,2.368242,0.482703,0.435502
2,2.251453,2.327417,0.504324,0.459391
3,2.132934,2.353421,0.508108,0.466976


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



MD2 training complete.

MD2 EPOCH VALIDATION HISTORY

Epoch: 1.0
  Validation loss: 2.368242
  Accuracy: 0.482703
  Macro-F1: 0.435502

Epoch: 2.0
  Validation loss: 2.327417
  Accuracy: 0.504324
  Macro-F1: 0.459391

Epoch: 3.0
  Validation loss: 2.353421
  Accuracy: 0.508108
  Macro-F1: 0.466976

Running final validation prediction...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



MD2 — OVERALL VALIDATION

Examples: 1850
Accuracy: 0.508108
Macro-F1: 0.466976

Per-class F1:
generic   : 0.592500
probing   : 0.375385
focus     : 0.591294
telling   : 0.308725

Class support:
generic   : 442
probing   : 412
focus     : 678
telling   : 318

Confusion Matrix:
Label order: ['generic', 'probing', 'focus', 'telling']
[[237  23 121  61]
 [ 49 122 193  48]
 [ 49  63 489  77]
 [ 23  30 173  92]]

MD2 — NON-TRUNCATED VALIDATION (ORIGINAL <=512 TOKENS)

Examples: 1507
Accuracy: 0.530192
Macro-F1: 0.464424

Per-class F1:
generic   : 0.648276
probing   : 0.400662
focus     : 0.606375
telling   : 0.202381

Class support:
generic   : 371
probing   : 368
focus     : 554
telling   : 214

Confusion Matrix:
Label order: ['generic', 'probing', 'focus', 'telling']
[[235  22  86  28]
 [ 49 121 172  26]
 [ 48  63 409  34]
 [ 22  30 128  34]]

MD2 — TRUNCATED VALIDATION (ORIGINAL >512 TOKENS)

Examples: 343
Accuracy: 0.411079
Macro-F1: 0.266889

Per-class F1:
generic   : 0.053333
probing 